In [1]:
# [CELL 1] STUDENT CONFIGURATION

from pathlib import Path

LAST_NAME = "Ayudtud"
STUDENT_ID = "TUPM-26-1939"

SEED_DIGIT = int(STUDENT_ID[-1])
ID_SUM = sum(int(d) for d in STUDENT_ID if d.isdigit())
NAME_LENGTH = len(LAST_NAME)

print("Seed:", SEED_DIGIT)
print("ID Sum:", ID_SUM)
print("Name Length:", NAME_LENGTH)

Seed: 9
ID Sum: 30
Name Length: 7


In [2]:
# [CELL 2] CREATE ACTIVITY WORKSPACE

WORK_DIR = Path(".")
DATA_DIR = WORK_DIR / "data"
OUTPUT_DIR = WORK_DIR / "output"
BACKUP_DIR = WORK_DIR / "backup"

DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)
BACKUP_DIR.mkdir(exist_ok=True)

print("Student:", LAST_NAME)
print("Student ID:", STUDENT_ID)
print("Workspace:", WORK_DIR.resolve())
print("Data folder exists:", DATA_DIR.exists())
print("Output folder exists:", OUTPUT_DIR.exists())
print("Backup folder exists:", BACKUP_DIR.exists())

Student: Ayudtud
Student ID: TUPM-26-1939
Workspace: C:\Users\Workstation\OneDrive\Lab6_File_Handling\Lab6_File_Handling_Additional_Exercises
Data folder exists: True
Output folder exists: True
Backup folder exists: True


In [3]:
# [CELL 3] CHECK DOWNLOADED DATASET

for file_path in DATA_DIR.iterdir():
    print(file_path.name)
    

weblog.csv


In [4]:
# [CELL 4] DEFINE DATASET PATH

LOG_FILE = DATA_DIR / "weblog.csv"

print("Dataset:", LOG_FILE)
print("File exists:", LOG_FILE.exists())

Dataset: data\weblog.csv
File exists: True


In [5]:
# [CELL 5] INSPECT CSV STRUCTURE

import csv

with open(LOG_FILE, "r", newline="", encoding="utf-8") as file:
    reader = csv.reader(file)

    print("Column Headers:", next(reader))

    print("\nFirst 5 Records:")
    for i, row in enumerate(reader):
        if i == 5:
            break
        print(row)

Column Headers: ['IP', 'Time', 'URL', 'Staus']

First 5 Records:
['10.128.2.1', '[29/Nov/2017:06:58:55', 'GET /login.php HTTP/1.1', '200']
['10.128.2.1', '[29/Nov/2017:06:59:02', 'POST /process.php HTTP/1.1', '302']
['10.128.2.1', '[29/Nov/2017:06:59:03', 'GET /home.php HTTP/1.1', '200']
['10.131.2.1', '[29/Nov/2017:06:59:04', 'GET /js/vendor/moment.min.js HTTP/1.1', '200']
['10.130.2.1', '[29/Nov/2017:06:59:06', 'GET /bootstrap-3.3.7/js/bootstrap.js HTTP/1.1', '200']


In [6]:
# [CELL 6] EXERCISE 1 CONFIGURATION

RECORDS_TO_PROCESS = 1000 + ID_SUM
TARGET_STATUS_CLASS = (SEED_DIGIT % 4) + 2

print("Records to process:", RECORDS_TO_PROCESS)
print("Target HTTP status class:", f"{TARGET_STATUS_CLASS}xx")

Records to process: 1030
Target HTTP status class: 3xx


In [7]:
# [CELL 7] INITIALIZE ANALYSIS VARIABLES

filtered_records = []
status_counts = {}

processed_count = 0
target_count = 0

print("Analysis variables initialized.")

Analysis variables initialized.


In [8]:
# [CELL 8] PROCESS WEB SERVER ACCESS LOG

with open(LOG_FILE, "r", newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)

    for row in reader:
        if processed_count >= RECORDS_TO_PROCESS:
            break

        processed_count += 1

        status_code = int(row["Staus"])

        if status_code // 100 == TARGET_STATUS_CLASS:
            filtered_records.append(row)
            status_counts[status_code] = status_counts.get(status_code, 0) + 1
            target_count += 1

print("Records processed:", processed_count)
print("Matching records:", target_count)
print("Status code counts:", status_counts)

Records processed: 1030
Matching records: 165
Status code counts: {302: 144, 304: 21}


In [9]:
# [CELL 9] COUNT URL REQUESTS

from collections import Counter

url_counts = Counter(
    row["URL"].split()[1]
    for row in filtered_records
)

top_5_urls = url_counts.most_common(5)

print("Top 5 Most Requested URLs:")
for url, count in top_5_urls:
    print(f"{url}: {count}")
    

Top 5 Most Requested URLs:
/: 47
/process.php: 35
/contestproblem.php?name=RUET%20OJ%20Server%20Testing%20Contest: 34
/home.php: 9
/logout.php: 7


In [10]:
# [CELL 10] EXPORT FILTERED RECORDS

filtered_csv_path = OUTPUT_DIR / "filtered_web_log.csv"

with open(filtered_csv_path, "w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=filtered_records[0].keys())
    writer.writeheader()
    writer.writerows(filtered_records)

print("Filtered CSV saved:", filtered_csv_path)
print("Exported records:", len(filtered_records))
print("File exists:", filtered_csv_path.exists())

Filtered CSV saved: output\filtered_web_log.csv
Exported records: 165
File exists: True


In [11]:
# [CELL 11] CREATE JSON SUMMARY

import json

summary = {
    "student": LAST_NAME,
    "student_id": STUDENT_ID,
    "records_processed": processed_count,
    "target_status_class": f"{TARGET_STATUS_CLASS}xx",
    "matching_records": target_count,
    "status_code_counts": status_counts,
    "top_5_urls": [
        {"url": url, "requests": count}
        for url, count in top_5_urls
    ]
}

summary_path = OUTPUT_DIR / "web_log_summary.json"

with open(summary_path, "w", encoding="utf-8") as file:
    json.dump(summary, file, indent=4)

print("JSON summary saved:", summary_path)
print("File exists:", summary_path.exists())

JSON summary saved: output\web_log_summary.json
File exists: True


In [12]:
# [CELL 12] BACKUP GENERATED FILES

import shutil

shutil.copy2(filtered_csv_path, BACKUP_DIR / filtered_csv_path.name)
shutil.copy2(summary_path, BACKUP_DIR / summary_path.name)

print("Backup files created:")
for file_path in BACKUP_DIR.iterdir():
    print(file_path.name)

Backup files created:
filtered_web_log.csv
web_log_summary.json


In [13]:
# [CELL 13] VERIFY BACKUP FILES

for file_name in ["filtered_web_log.csv", "web_log_summary.json"]:
    original = OUTPUT_DIR / file_name
    backup = BACKUP_DIR / file_name

    print(
        file_name,
        "Verified:",
        original.exists() and backup.exists() and original.read_bytes() == backup.read_bytes()
    )

filtered_web_log.csv Verified: True
web_log_summary.json Verified: True
